# Property Scanner: Multimodal AI Valuation System

## 1. Project Overview
**Objective**: Predict property prices in Spanish cities (Madrid, Barcelona, etc.) using a **Multimodal Late-Fusion Model**.

**Key Innovation**: Unlike traditional comparable approaches that only use numbers (sqm, rooms), this system "sees" the property using a **Vision Language Model (VLM)** to extract qualitative features (condition, style, finishes) from images and fuses them with quantitative data.


## 2. System Architecture

The system uses a state-of-the-art **Agentic Pipeline** to transform raw web data into reasoned valuations.

### Pipeline Stages:
1.  **Extraction Agent**: Scrapes and normalizes raw data into `CanonicalListing` objects.
2.  **Vision Agent (VLM)**: Deep-inspects photos using **Llava/Ollama** to describe interiors and finishes.
3.  **Embedding Engine**: Converts text and descriptions into **384D vectors** via SentenceTransformers.
4.  **Fusion Model**: A custom PyTorch model that performs **Cross-Attention** between the target property and local market comparables.


## 3. Data Inspection

Let's look at the data collected in `data/listings.db`.

In [ ]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import ast
import json
import seaborn as sns
import sys
import os

# Ensure we can import from src
sys.path.append('..')

# Connect to Database
conn = sqlite3.connect('../data/listings.db')
df = pd.read_sql_query("SELECT * FROM listings WHERE price > 0", conn)
conn.close()

print(f"Total Listings: {len(df)}")
display(df[['id', 'title', 'price', 'city', 'vlm_description']].head())

### 3.1 Market Composition: City Breakdown
Where is our data coming from? We've expanded to include the most active real estate markets in Spain.

In [ ]:
plt.figure(figsize=(10, 6))
city_counts = df['city'].value_counts()
sns.barplot(x=city_counts.index, y=city_counts.values, palette='viridis')
plt.title('Listings per City')
plt.xticks(rotation=45)
plt.ylabel('Count')
plt.show()

### 3.2 Geospatial Analysis
Visualizing the spatial density of our properties.

In [ ]:
# Filter out valid coordinates
geo_df = df[(df['lat'] != 0) & (df['lon'] != 0)]

plt.figure(figsize=(12, 8))
plt.scatter(geo_df['lon'], geo_df['lat'], c=geo_df['price'], cmap='magma', s=15, alpha=0.6)
plt.colorbar(label='Price (€)')
plt.title('Property Distribution Across Spain')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

## 4. VLM: The Visual Intelligence

Traditional models miss detail. Our VLM captures descriptions like *"modern open-concept kitchen"* or *"ranch-style villa with pool"*.

In [ ]:
vlm_enriched = df[df['vlm_description'].notna() & (df['vlm_description'] != '')]
print(f"Enriched Listings: {len(vlm_enriched)}")

if not vlm_enriched.empty:
    sample = vlm_enriched.sample(1).iloc[0]
    print(f"\nProperty: {sample['title']}")
    print("VLM Insight:", sample['vlm_description'][:400] + "...")

## 5. Model Training & Performance

Below is the current training convergence for the **92k-parameter compact model**.

In [ ]:
checkpoint_path = '../models/checkpoint_latest.pt'
if os.path.exists(checkpoint_path):
    ckpt = torch.load(checkpoint_path, map_location='cpu')
    plt.figure(figsize=(10, 5))
    plt.plot(ckpt['train_losses'], label='Train')
    plt.plot(ckpt['val_losses'], label='Val')
    plt.title('Model Training Convergence (Quantile Loss)')
    plt.legend()
    plt.show()
else:
    print("Checkpoint not found. Run training first.")

## 6. Live Inference Example

Compare the AI's valuation with the actual market price.

In [ ]:
from src.training.dataset import PropertyDataset, collate_fn
import torch
import numpy as np

# Set up dataset and sample
ds = PropertyDataset(db_path='../data/listings.db', use_vlm=True)
idx = np.random.randint(0, len(ds))
item = ds[idx]
batch = collate_fn([item])

# Load Model
model = PropertyFusionModel(hidden_dim=64, num_heads=2) # Compact settings
if os.path.exists('../models/fusion_model.pt'):
    model.load_state_dict(torch.load('../models/fusion_model.pt', map_location='cpu'))
    model.eval()
    with torch.no_grad():
        q, _, _, _ = model(
            target_tab=batch['target_tab'], 
            target_text=batch['target_text'],
            comp_tab=batch['comp_tab'], 
            comp_text=batch['comp_text'], 
            comp_prices=batch['comp_prices']
        )
        
    actual = batch['target_price'][0].item()
    pred = q[0, 1].item()
    print(f"Title: {ds.listings[idx]['title']}")
    print(f"Actual: €{actual:,.0f} | Predicted: €{pred:,.0f} | Error: {abs(actual-pred)/actual*100:.1f}%")
else:
    print("Model weights not found. Prediction skipped.")